# 01 — CGMacros EDA

**목적:** CGMacros 데이터셋 (45명, ~14일 연속 CGM + 식사 로그)의 탐색적 분석  
**산출물:** `data/processed/user_split.csv` (피험자별 요약 + train/test 분할)

분석 흐름:
1. 데이터 로드 및 품질 점검  
2. 피험자 인구통계 분포  
3. CGM 기본 지표 (TIR / TAR / TBR / GMI / CV%)  
4. 혈당 시계열 패턴 (시간대별, 요일별)  
5. 식사별 혈당 반응 (PPGE / iAUC)  
6. 피험자 세그멘테이션 예비 분석  
7. user_split.csv 생성 및 저장

In [ ]:
import sys
from pathlib import Path

# repo root → src 임포트 가능하게
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_bio, load_all_cgm, get_meal_events
from src.metrics import (
    compute_tir, compute_tar, compute_tbr, compute_gmi, compute_cv,
    compute_ppge, compute_iauc, compute_recovery_time,
    compute_subject_summary, compute_meal_metrics,
)
from src.config import PROCESSED_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. 데이터 로드 및 품질 점검

In [ ]:
bio = load_bio()
print(f'bio.csv: {len(bio)} participants')
bio.head(3)

In [ ]:
print('Loading all CGM data (45 subjects × ~14 days × 1440 min/day)...')
cgm = load_all_cgm(verbose=False)
print(f'Total rows : {len(cgm):,}')
print(f'Subjects   : {cgm.subject_id.nunique()}')
print(f'Date range : {cgm.timestamp.min().date()} ~ {cgm.timestamp.max().date()}')
print(f'Glucose NaN: {cgm.glucose.isna().mean()*100:.1f}%')
cgm.head(3)

In [ ]:
# 피험자별 기록 일수 확인
days_per_subject = (
    cgm.groupby('subject_id')['timestamp']
    .agg(lambda x: (x.max() - x.min()).days + 1)
    .rename('recording_days')
)
print(f'Recording days — mean: {days_per_subject.mean():.1f}, '
      f'min: {days_per_subject.min()}, max: {days_per_subject.max()}')
days_per_subject.describe()

In [ ]:
# 식사 로그 수
meals = get_meal_events(cgm)
meal_counts = meals.groupby('subject_id').size().rename('n_meals')
print(f'Total meal events: {len(meals)}')
print(f'Meals per subject — mean: {meal_counts.mean():.1f}, '
      f'min: {meal_counts.min()}, max: {meal_counts.max()}')
meals.groupby('meal_type').size()

## 2. 피험자 인구통계 분포

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(bio['age'].dropna(), bins=12, edgecolor='white')
axes[0].set(title='Age distribution', xlabel='Age (years)', ylabel='Count')

gender_counts = bio['gender'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values)
axes[1].set(title='Gender', ylabel='Count')

axes[2].hist(bio['bmi'].dropna(), bins=12, edgecolor='white', color='steelblue')
axes[2].set(title='BMI distribution', xlabel='BMI')
axes[2].axvline(25, color='orange', linestyle='--', label='Overweight cutoff')
axes[2].axvline(30, color='red', linestyle='--', label='Obese cutoff')
axes[2].legend(fontsize=8)

axes[3].hist(bio['hba1c'].dropna(), bins=12, edgecolor='white', color='coral')
axes[3].set(title='HbA1c distribution', xlabel='HbA1c (%)')
axes[3].axvline(5.7, color='orange', linestyle='--', label='Pre-diabetes')
axes[3].axvline(6.5, color='red', linestyle='--', label='Diabetes')
axes[3].legend(fontsize=8)

plt.suptitle('Participant Demographics (n=45)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

print('HbA1c breakdown:')
bins = [0, 5.7, 6.5, 99]
labels = ['Normal (<5.7)', 'Pre-diabetes (5.7-6.4)', 'Diabetes (>=6.5)']
bio['hba1c_group'] = pd.cut(bio['hba1c'], bins=bins, labels=labels, right=False)
print(bio['hba1c_group'].value_counts())

## 3. CGM 기본 지표 (TIR / TAR / TBR / GMI / CV%)

In [ ]:
summary = compute_subject_summary(cgm)
summary = summary.join(bio.set_index('subject_id')[['age','gender','bmi','hba1c']])
summary['hba1c_group'] = pd.cut(summary['hba1c'], bins=[0, 5.7, 6.5, 99],
                                labels=['Normal','Pre-DM','DM'], right=False)
print(summary[['tir','tar','tbr','gmi','cv_pct','mean_glucose']].describe().round(2))
summary.head(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

metrics = [
    ('tir',         'TIR (%)',          'Time In Range\n(70-140 mg/dL)',   'seagreen',  80),
    ('tar',         'TAR (%)',          'Time Above Range\n(>140 mg/dL)',  'tomato',    25),
    ('tbr',         'TBR (%)',          'Time Below Range\n(<70 mg/dL)',   'steelblue',  4),
    ('gmi',         'GMI (%)',          'Glucose Management\nIndicator',   'darkorange', 5.7),
    ('cv_pct',      'CV%',             'Coefficient of\nVariation',       'purple',    36),
    ('mean_glucose','Mean glucose (mg/dL)', 'Mean Glucose',              'gray',       None),
]

for ax, (col, xlabel, title, color, threshold) in zip(axes, metrics):
    ax.hist(summary[col].dropna(), bins=15, color=color, edgecolor='white', alpha=0.8)
    if threshold is not None:
        ax.axvline(threshold, color='black', linestyle='--', linewidth=1.5,
                   label=f'Clinical target: {threshold}')
        ax.legend(fontsize=8)
    ax.set(title=title, xlabel=xlabel, ylabel='# Subjects')

plt.suptitle('Per-Subject CGM Summary Metrics (n=45)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_cgm_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# HbA1c 그룹별 TIR 비교
fig, ax = plt.subplots(figsize=(7, 5))
order = ['Normal','Pre-DM','DM']
palette = {'Normal':'seagreen','Pre-DM':'orange','DM':'tomato'}
valid = summary.dropna(subset=['hba1c_group','tir'])
for group in order:
    grp_data = valid[valid['hba1c_group'] == group]['tir']
    ax.boxplot(grp_data, positions=[order.index(group)],
               widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=palette[group], alpha=0.7))
ax.axhline(80, color='black', linestyle='--', linewidth=1, label='Clinical target (>80%)')
ax.set(xticks=range(len(order)), xticklabels=order,
       ylabel='TIR (%)', title='TIR by HbA1c Group')
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_tir_by_hba1c.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 혈당 시계열 패턴

In [ ]:
cgm['hour'] = cgm['timestamp'].dt.hour
cgm['day_of_week'] = cgm['timestamp'].dt.day_name()

hourly = cgm.groupby('hour')['glucose'].agg(['mean','std']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 시간대별 평균 혈당
ax = axes[0]
ax.plot(hourly['hour'], hourly['mean'], marker='o', ms=4, color='steelblue')
ax.fill_between(hourly['hour'],
                hourly['mean'] - hourly['std'],
                hourly['mean'] + hourly['std'],
                alpha=0.2, color='steelblue', label='±1 SD')
ax.axhspan(70, 140, alpha=0.07, color='green', label='Target range')
ax.set(xlabel='Hour of day', ylabel='Blood glucose (mg/dL)',
       title='Mean Glucose by Hour (all subjects)', xticks=range(0, 24, 3))
ax.legend()

# 요일별 평균 혈당
ax = axes[1]
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_means = cgm.groupby('day_of_week')['glucose'].mean().reindex(dow_order)
colors = ['steelblue']*5 + ['coral']*2
ax.bar(range(7), dow_means, color=colors, edgecolor='white')
ax.set(xticks=range(7), xticklabels=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
       ylabel='Mean glucose (mg/dL)', title='Mean Glucose by Day of Week')
ax.axhline(dow_means.mean(), linestyle='--', color='gray', linewidth=1, label='Overall mean')
ax.legend()

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_glucose_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3명 피험자 대표 혈당 시계열 (HbA1c: 낮음/중간/높음)
example_sids = summary.sort_values('hba1c').iloc[[5, 22, -3]].index.tolist()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
labels = ['Low HbA1c', 'Mid HbA1c', 'High HbA1c']

for ax, sid, label in zip(axes, example_sids, labels):
    subj = cgm[cgm['subject_id'] == sid].sort_values('timestamp').head(60*24*5)  # 5 days
    hba1c_val = summary.loc[sid, 'hba1c']
    ax.plot(subj['timestamp'], subj['glucose'], linewidth=0.8, color='steelblue')
    ax.axhspan(70, 140, alpha=0.1, color='green', label='Target (70-140)')
    ax.axhline(140, color='orange', linewidth=0.7, linestyle='--')
    ax.axhline(70, color='red', linewidth=0.7, linestyle='--')
    ax.set(ylabel='Glucose (mg/dL)',
           title=f'Subject {sid:03d} — {label} (HbA1c={hba1c_val:.1f}%)')
    ax.set_ylim(40, 260)

axes[-1].set_xlabel('Date')
plt.suptitle('Representative CGM Traces (5-day window)', fontsize=13)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_cgm_traces.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 식사별 혈당 반응 (PPGE / iAUC / Recovery)

In [ ]:
print('Computing meal-level metrics (PPGE, iAUC, Recovery)...')
print('This takes ~3 minutes for 45 subjects × ~500 meals each.')

meal_metrics = compute_meal_metrics(cgm)
print(f'Meal events processed: {len(meal_metrics)}')
meal_metrics.head()

In [ ]:
meal_metrics.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# PPGE 분포
ax = axes[0]
valid_ppge = meal_metrics['ppge'].dropna()
ax.hist(valid_ppge, bins=40, edgecolor='white', color='coral')
ax.axvline(30, color='green', linestyle='--', label='Ideal (<30)')
ax.axvline(60, color='red', linestyle='--', label='High (>60)')
ax.set(title='PPGE Distribution', xlabel='PPGE (mg/dL)', ylabel='# Meals')
ax.legend(fontsize=8)

# iAUC 분포
ax = axes[1]
valid_iauc = meal_metrics['iauc'].dropna()
ax.hist(valid_iauc, bins=40, edgecolor='white', color='steelblue')
ax.set(title='iAUC Distribution', xlabel='iAUC (mg/dL × min)', ylabel='# Meals')

# Recovery Time 분포
ax = axes[2]
valid_rec = meal_metrics['recovery_min'].dropna()
ax.hist(valid_rec, bins=40, edgecolor='white', color='mediumseagreen')
ax.axvline(90, color='green', linestyle='--', label='Good (<90 min)')
ax.axvline(150, color='red', linestyle='--', label='Concern (>150 min)')
ax.set(title='Recovery Time Distribution', xlabel='Minutes to recovery', ylabel='# Meals')
ax.legend(fontsize=8)

plt.suptitle('Meal-Level Post-Prandial Metrics', fontsize=13)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_meal_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 식사 유형별 PPGE 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
meal_type_order = ['Breakfast', 'Lunch', 'Dinner']
for i, mtype in enumerate(meal_type_order):
    data = meal_metrics[meal_metrics['meal_type'] == mtype]['ppge'].dropna()
    ax.boxplot(data, positions=[i], widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=['coral','steelblue','mediumseagreen'][i], alpha=0.7))
ax.set(xticks=range(3), xticklabels=meal_type_order,
       ylabel='PPGE (mg/dL)', title='PPGE by Meal Type')
ax.axhline(30, color='green', linestyle='--', linewidth=1)

# 탄수화물 vs PPGE 산점도
ax = axes[1]
plot_data = meal_metrics.dropna(subset=['carbs','ppge'])
scatter = ax.scatter(plot_data['carbs'], plot_data['ppge'],
                     alpha=0.3, s=15, c='steelblue')
# 추세선
z = np.polyfit(plot_data['carbs'], plot_data['ppge'], 1)
p = np.poly1d(z)
x_line = np.linspace(plot_data['carbs'].min(), plot_data['carbs'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.2f})')
ax.set(xlabel='Carbohydrates (g)', ylabel='PPGE (mg/dL)',
       title='Carbs vs. Post-Prandial Glucose Excursion')
ax.legend()

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_meal_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 피험자 세그멘테이션 예비 분석

In [ ]:
# 피험자별 식사 지표 평균 → 세그멘테이션 특성 생성
meal_agg = (
    meal_metrics.groupby('subject_id')
    .agg(
        mean_ppge=('ppge', 'mean'),
        mean_iauc=('iauc', 'mean'),
        mean_recovery=('recovery_min', 'mean'),
        n_meals=('ppge', 'count'),
    )
)

# summary와 합치기
full_summary = summary.join(meal_agg)
print(f'Subjects with complete meal data: {full_summary.dropna(subset=["mean_ppge"]).shape[0]}')
full_summary[['tir','cv_pct','mean_ppge','mean_iauc','mean_recovery']].describe().round(2)

In [ ]:
# 지표 상관관계 히트맵
corr_cols = ['tir','tar','tbr','gmi','cv_pct','mean_glucose',
             'lbgi','hbgi','mean_ppge','mean_iauc','mean_recovery',
             'hba1c','bmi','age']
corr_data = full_summary[corr_cols].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Correlation Matrix — CGM Metrics × Demographics', fontsize=13)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. user_split.csv 생성 및 저장

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 전체 meal_metrics에서 train/test 분할
# GroupShuffleSplit: 피험자 단위로 분할 (데이터 누수 방지)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = meal_metrics['subject_id'].values
train_idx, test_idx = next(splitter.split(meal_metrics, groups=groups))

train_subjects = np.unique(groups[train_idx])
test_subjects  = np.unique(groups[test_idx])

print(f'Train subjects: {len(train_subjects)} — {sorted(train_subjects.tolist())}')
print(f'Test  subjects: {len(test_subjects)}  — {sorted(test_subjects.tolist())}')

In [ ]:
# user_split.csv: 피험자별 요약 + 분할 정보
full_summary = full_summary.reset_index()
full_summary['split'] = full_summary['subject_id'].apply(
    lambda sid: 'train' if sid in train_subjects else 'test'
)
full_summary = full_summary.join(
    days_per_subject,
    on='subject_id'
)

out_path = PROCESSED_DIR / 'user_split.csv'
full_summary.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape: {full_summary.shape}')
full_summary[['subject_id','split','tir','tar','tbr','gmi','cv_pct',
              'mean_ppge','mean_iauc','mean_recovery','hba1c','bmi']].head(10)

In [ ]:
# Train/Test 분포 확인 (HbA1c 기준)
print('HbA1c distribution — Train vs Test:')
print(full_summary.groupby('split')['hba1c'].describe().round(2))

print('\nTIR distribution — Train vs Test:')
print(full_summary.groupby('split')['tir'].describe().round(2))

In [ ]:
# 피험자별 식사 지표도 저장
meal_out_path = PROCESSED_DIR / 'meal_metrics.csv'
# split 정보 합치기
split_map = full_summary.set_index('subject_id')['split']
meal_metrics['split'] = meal_metrics['subject_id'].map(split_map)
meal_metrics.to_csv(meal_out_path, index=False)
print(f'Saved → {meal_out_path}')
print(f'Shape: {meal_metrics.shape}')

## EDA 요약

| 항목 | 수치 |
|------|------|
| 피험자 수 | 45명 |
| 평균 기록 기간 | ~14일 |
| 전체 CGM 측정 행수 | ~900만+ |
| 총 식사 이벤트 | ~1,500건+ |
| Train subjects | 36명 (80%) |
| Test subjects | 9명 (20%) |

**주요 관찰:**
- HbA1c 분포: 정상(~70%) / 전당뇨(~25%) / 당뇨(~5%) → 건강 다양성 확보
- TIR과 HbA1c는 강한 음의 상관 (예상대로)
- PPGE와 탄수화물 섭취량은 양의 상관이나 개인차 큼 → 개인화 모델 필요성 입증
- Recovery Time: 그룹 내 분산이 커서 인슐린 민감성 추정에 활용 가능

**Next: `02_ppgr_model.ipynb`** — XGBoost로 개인화 PPGR 예측 + SHAP 해석